# Laboratorio resuelto: métricas para nowcasting de precipitación

Este notebook queda como referencia del instructor o solución completa. Usa las utilidades compartidas en `course_utils/` para cargar datos, aplicar la paleta común, graficar y calcular métricas.

Trabajaremos con secuencias `(25, 128, 128)` en mm/h:

- `inputs = sequence[:13]`
- `target = sequence[13:25]`
- `prediction = CasCast/EarthFormer/modelo si existe; si no, persistencia`

## 0. Preparación

Si faltan datos, ejecuta desde `nowcasting_course_lab/`:

```bash
python scripts/download_assets.py
python scripts/make_persistence_predictions.py
```

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cwd = Path.cwd().resolve()
if cwd.name == "notebooks":
    PROJECT_ROOT = cwd.parent
else:
    PROJECT_ROOT = cwd
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from course_utils.data import (
    SAMPLE_FILES,
    LEAD_MINUTES,
    get_paths,
    load_sample,
    split_sequence,
    load_prediction,
    make_persistence_prediction,
)
from course_utils.metrics import (
    THRESHOLDS,
    THRESHOLD_LABELS,
    continuous_metrics,
    binary_metrics,
    event_metrics_by_threshold_and_lead,
    summarize_events_by_threshold,
)
from course_utils.palette import apply_course_style
from course_utils.plotting import (
    plot_event_grid,
    plot_target_prediction_panel,
    plot_rmse_bias,
    plot_csi_by_threshold,
)

apply_course_style()
paths = get_paths(PROJECT_ROOT)
print(f"Proyecto: {paths.root}")
print(f"Muestras: {paths.sample_dir}")

## 1. Cargar una secuencia

In [ ]:
sample_name = SAMPLE_FILES[0]
sequence = load_sample(sample_name, paths)
inputs, target = split_sequence(sequence)
prediction, prediction_source = load_prediction(sample_name, inputs, paths)

print("Archivo:", sample_name)
print("Fuente de predicción:", prediction_source)
print("sequence:", sequence.shape)
print("inputs:", inputs.shape)
print("target:", target.shape)
print("prediction:", prediction.shape)
print("Rango secuencia mm/h:", float(sequence.min()), "a", float(sequence.max()))

## Tarea 1: visualizar el problema

Observa desplazamiento, núcleos intensos, suavizado y degradación con el tiempo de pronóstico.

In [ ]:
fig = plot_event_grid(inputs, target, prediction, sample_name, prediction_source)
plt.show()

## Tarea 2: métricas continuas

Calculamos MAE, RMSE, bias y correlación de Pearson por lead time.

In [ ]:
continuous_df = continuous_metrics(prediction, target)
continuous_df

In [ ]:
fig = plot_rmse_bias(continuous_df)
plt.show()

## Tarea 3: métricas de eventos

Usamos umbrales de lluvia para transformar los campos continuos en eventos binarios.

In [ ]:
event_df = event_metrics_by_threshold_and_lead(prediction, target, THRESHOLDS)
event_df.head(12)

In [ ]:
summary_df = summarize_events_by_threshold(event_df)
summary_df

In [ ]:
fig = plot_csi_by_threshold(event_df)
plt.show()

## Momento clave: RMSE vs CSI de lluvia intensa

Una predicción puede verse razonable en RMSE y aun así perder habilidad para lluvia fuerte o intensa.

In [ ]:
comparison = continuous_df[["lead_min", "RMSE", "Bias"]].merge(
    event_df[event_df["threshold_mm_h"].isin([0.5, 10.0])][["lead_min", "threshold_mm_h", "CSI"]]
    .pivot(index="lead_min", columns="threshold_mm_h", values="CSI")
    .rename(columns={0.5: "CSI_0.5", 10.0: "CSI_10.0"})
    .reset_index(),
    on="lead_min",
)
comparison

## Panel compacto estilo reporte

Este panel usa la misma paleta compartida para comparar varios casos, similar a una figura de resultados.

In [ ]:
cases = []
for name in SAMPLE_FILES[:3]:
    seq = load_sample(name, paths)
    x, y = split_sequence(seq)
    pred, src = load_prediction(name, x, paths)
    cases.append({"label": name.split("_rain_rate")[0], "target": y, "prediction": pred})

fig = plot_target_prediction_panel(cases, title="Objetivo vs predicción: ejemplos IDEAM")
plt.show()

## Reflexión final

¿Por qué un modelo puede tener bajo RMSE pero baja utilidad para nowcasting de lluvia intensa?

Una buena respuesta debe mencionar que el suavizado reduce error promedio pixel a pixel, pero puede eliminar núcleos intensos y localizados. Por eso modelos como CasCast combinan una parte determinística para movimiento/estructura amplia con una etapa probabilística para detalles de pequeña escala.